# Deep dive 8/13: `call_center_interactions`

Second fact table. Dictionary declares 800,000 rows, Source: Contact Center, Partition: daily.

`table_specs.py` has a specific open question baked into its comment above this spec: **`contact_reason` and `reason_category` showed byte-for-byte identical value distributions in the original Bronze profile** (same 6 categories, same row counts down to the last digit) -- strong evidence they're the same value stored twice, but the profile only confirmed matching *aggregate* distributions, not a row-by-row identity check. The comment literally asks for `SELECT count(*) FROM silver.fact_call_center_interactions WHERE contact_reason <> reason_category` before trusting either is safe to drop -- that's section 7 here.

Dictionary NOT NULL: `interaction_id`, `interaction_date`, `process_date`, `customer_id`, `interaction_type`, `channel`, `contact_reason`, `reason_category`, `requires_followup`, `was_escalated`, `has_transcript`, `has_recording`. `contracts.py`'s current `required` tuple has 11 of these 12 -- **`has_recording` is missing**, checked as a fix candidate in section 3. Nullable: `agent_id` (a nullable FK -- not every interaction has an assigned agent), `duration_seconds`, `wait_time_seconds`, `was_resolved`, `detected_sentiment`, `sentiment_score`, `customer_detected_accent`, `agent_used_accent`, `mentioned_products`.

`channel` is already gated in `contracts.py` with 6 values (`Phone, Web Chat, WhatsApp, Email, App, Web`) -- one more than the dictionary's declared 5 (`Phone, Web Chat, WhatsApp, Email, App`). Section 5 confirms whether the extra `Web` value is real and whether the existing gate is still accurate.

Run top to bottom. Checklist near the end; free cells after that.

In [ ]:
import duckdb
import pandas as pd

pd.set_option("display.max_rows", 1000)
pd.set_option("display.min_rows", 1000)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 100)

con = duckdb.connect("data/latam_bank.duckdb")

def q(sql, params=None):
    """Run a query and return a pandas DataFrame."""
    return con.execute(sql, params or []).df()

def q_full(sql, params=None):
    """Like q(), but shows every cell's full text with no truncation."""
    with pd.option_context("display.max_colwidth", None):
        return q(sql, params)

## 0. Sample rows -- Bronze and Silver side by side

In [ ]:
sample_ids = q("SELECT interaction_id FROM bronze.call_center_interactions ORDER BY interaction_id LIMIT 20")["interaction_id"].tolist()
ids_sql = ", ".join(f"'{i}'" for i in sample_ids)

print("Bronze:")
display(q_full(f"SELECT * FROM bronze.call_center_interactions WHERE interaction_id IN ({ids_sql}) ORDER BY interaction_id"))

In [ ]:
print("Silver:")
display(q_full(f"SELECT * FROM silver.fact_call_center_interactions WHERE interaction_id IN ({ids_sql}) ORDER BY interaction_id"))

## 1. Column inventory -- names AND types, Bronze vs. Silver

In [ ]:
expected_silver_columns = [
    "interaction_id", "interaction_date", "process_date", "customer_id", "agent_id",
    "interaction_type", "channel", "contact_reason", "reason_category", "duration_seconds",
    "wait_time_seconds", "was_resolved", "requires_followup", "detected_sentiment",
    "sentiment_score", "customer_detected_accent", "agent_used_accent", "was_escalated",
    "mentioned_products", "has_transcript", "has_recording",
]

actual_silver_columns = q("""
    SELECT column_name FROM information_schema.columns
    WHERE table_schema='silver' AND table_name='fact_call_center_interactions'
    ORDER BY ordinal_position
""")["column_name"].tolist()

missing = [c for c in expected_silver_columns if c not in actual_silver_columns]
extra = [c for c in actual_silver_columns if c not in expected_silver_columns]
print("Missing from silver.fact_call_center_interactions:", missing or "none")
print("Extra in silver.fact_call_center_interactions:", extra or "none")
print()
print("Column order as built:", actual_silver_columns)

In [ ]:
print("Bronze columns + types:")
display(q("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema='bronze' AND table_name='call_center_interactions' ORDER BY ordinal_position
"""))

print("Silver columns + types:")
display(q("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema='silver' AND table_name='fact_call_center_interactions' ORDER BY ordinal_position
"""))

## 2. Row-count reconciliation + dedup

In [ ]:
bronze_total = q("SELECT count(*) AS n FROM bronze.call_center_interactions")["n"][0]
bronze_distinct_pk = q("SELECT count(DISTINCT interaction_id) AS n FROM bronze.call_center_interactions")["n"][0]
silver_total = q("SELECT count(*) AS n FROM silver.fact_call_center_interactions")["n"][0]
silver_distinct_pk = q("SELECT count(DISTINCT interaction_id) AS n FROM silver.fact_call_center_interactions")["n"][0]

print(f"bronze.call_center_interactions:      {bronze_total:,} rows, {bronze_distinct_pk:,} distinct interaction_id")
print(f"silver.fact_call_center_interactions: {silver_total:,} rows, {silver_distinct_pk:,} distinct interaction_id")
print()
print("Dictionary declares 800,000 rows -- matches" if bronze_total == 800_000 else f"Dictionary declares 800,000 rows -- MISMATCH, Bronze has {bronze_total:,}")
print("Duplicate raw rows in Bronze:", bronze_total - bronze_distinct_pk)
print("OK -- Silver row count matches Bronze distinct PK count" if silver_total == bronze_distinct_pk
      else "MISMATCH -- investigate")

## 3. NOT NULL audit -- Bronze raw vs. Silver typed, all 12 dictionary-declared columns

Including `has_recording`, missing from `contracts.py`'s `required` tuple -- checked here as a fix candidate.

In [ ]:
declared_not_null = [
    "interaction_id", "interaction_date", "process_date", "customer_id", "interaction_type",
    "channel", "contact_reason", "reason_category", "requires_followup", "was_escalated",
    "has_transcript", "has_recording",
]
currently_required = {"interaction_id","interaction_date","process_date","customer_id","interaction_type",
                       "channel","contact_reason","reason_category","requires_followup","was_escalated","has_transcript"}

rows = []
for col in declared_not_null:
    bronze_nulls = q(f"""
        SELECT count(*) AS n FROM bronze.call_center_interactions
        WHERE {col} IS NULL OR trim(CAST({col} AS VARCHAR)) = ''
    """)["n"][0]
    silver_nulls = q(f"""
        SELECT count(*) AS n FROM silver.fact_call_center_interactions WHERE {col} IS NULL
    """)["n"][0]
    rows.append({"column": col, "declared": "NOT NULL", "in_contracts_required": col in currently_required,
                 "bronze_nulls": bronze_nulls, "silver_nulls": silver_nulls,
                 "cast_introduced_nulls": silver_nulls - bronze_nulls})

pd.DataFrame(rows)

## 4. Nullable columns -- population rate, Bronze vs. Silver

In [ ]:
nullable_cols = ["agent_id", "duration_seconds", "wait_time_seconds", "was_resolved",
                  "detected_sentiment", "sentiment_score", "customer_detected_accent",
                  "agent_used_accent", "mentioned_products"]
rows = []
total = q("SELECT count(*) AS n FROM bronze.call_center_interactions")["n"][0]
for col in nullable_cols:
    bronze_populated = q(f"""
        SELECT count(*) AS n FROM bronze.call_center_interactions
        WHERE {col} IS NOT NULL AND trim(CAST({col} AS VARCHAR)) <> ''
    """)["n"][0]
    silver_populated = q(f"""
        SELECT count(*) AS n FROM silver.fact_call_center_interactions WHERE {col} IS NOT NULL
    """)["n"][0]
    rows.append({"column": col, "total": total,
                 "bronze_populated": bronze_populated, "bronze_pct": round(100*bronze_populated/total, 2),
                 "silver_populated": silver_populated, "silver_pct": round(100*silver_populated/total, 2)})

pd.DataFrame(rows)

## 5. Domain checks -- `interaction_type`, `reason_category`, `detected_sentiment`, and re-verifying the already-gated `channel`

In [ ]:
# interaction_type -- dictionary: Inbound Call, Outbound Call, Chat, Email, Video
print("Bronze:")
display(q("SELECT interaction_type, count(*) AS n FROM bronze.call_center_interactions GROUP BY 1 ORDER BY n DESC"))
print("Silver:")
display(q("SELECT interaction_type, count(*) AS n FROM silver.fact_call_center_interactions GROUP BY 1 ORDER BY n DESC"))

In [ ]:
# reason_category -- dictionary text was truncated in the PDF table, so discovering the real
# domain directly rather than assuming.
print("Bronze:")
display(q("SELECT reason_category, count(*) AS n FROM bronze.call_center_interactions GROUP BY 1 ORDER BY n DESC"))

In [ ]:
# detected_sentiment -- dictionary: Positive, Neutral, Negative, Very Negative
print("Bronze:")
display(q("SELECT detected_sentiment, count(*) AS n FROM bronze.call_center_interactions GROUP BY 1 ORDER BY n DESC"))

In [ ]:
# channel -- already gated in contracts.py as {Phone, Web Chat, WhatsApp, Email, App, Web}
# (6 values -- one more than the dictionary's declared 5). Confirming the gate is still accurate.
gated_channel_domain = {"Phone", "Web Chat", "WhatsApp", "Email", "App", "Web"}
real_channel = set(q("SELECT DISTINCT channel AS v FROM bronze.call_center_interactions")["v"])
print("channel values NOT covered by the current contracts.py domain:", real_channel - gated_channel_domain or "none -- gate is still accurate")
display(q("SELECT channel, count(*) AS n FROM bronze.call_center_interactions GROUP BY 1 ORDER BY n DESC"))

## 6. FK integrity -- `customer_id` (required) and `agent_id` (nullable), reproduced directly

In [ ]:
fk_checks = [
    ("customer_id", "silver.dim_customers", "customer_id"),
    ("agent_id", "silver.dim_service_agents", "agent_id"),
]
rows = []
for child_col, parent_table, parent_col in fk_checks:
    total = q(f"SELECT count(*) AS n FROM silver.fact_call_center_interactions WHERE {child_col} IS NOT NULL")["n"][0]
    orphans = q(f"""
        SELECT count(*) AS n FROM silver.fact_call_center_interactions t
        WHERE t.{child_col} IS NOT NULL
          AND NOT EXISTS (SELECT 1 FROM {parent_table} p WHERE p.{parent_col} = t.{child_col})
    """)["n"][0]
    rows.append({"fk": f"{child_col} -> {parent_table}.{parent_col}", "populated": total,
                 "orphans": orphans, "orphan_pct": round(100*orphans/total, 4) if total else None})

pd.DataFrame(rows)

## 7. `contact_reason` vs. `reason_category` -- the exact identity check `table_specs.py`'s comment asks for

The comment above the spec flags matching aggregate distributions between these two columns as suspicious but unconfirmed at the row level. Running the exact query it names.

In [ ]:
mismatch_count = q("SELECT count(*) AS n FROM silver.fact_call_center_interactions WHERE contact_reason <> reason_category")["n"][0]
total = q("SELECT count(*) AS n FROM silver.fact_call_center_interactions")["n"][0]
print(f"Rows where contact_reason <> reason_category: {mismatch_count:,} / {total:,}")
print("These ARE the same value stored twice" if mismatch_count == 0 else "These are NOT identical -- some real divergence exists")

In [ ]:
# If there's any divergence, show a sample of the mismatching rows to see the pattern.
q_full("""
    SELECT interaction_id, contact_reason, reason_category
    FROM silver.fact_call_center_interactions
    WHERE contact_reason <> reason_category
    LIMIT 20
""")

## 8. `duration_seconds`/`wait_time_seconds` -- range sanity

In [ ]:
q("""
    SELECT
        typeof(duration_seconds) AS duration_type, typeof(wait_time_seconds) AS wait_type,
        min(duration_seconds) AS min_duration, max(duration_seconds) AS max_duration,
        min(wait_time_seconds) AS min_wait, max(wait_time_seconds) AS max_wait,
        count(*) FILTER (WHERE duration_seconds < 0) AS negative_duration,
        count(*) FILTER (WHERE wait_time_seconds < 0) AS negative_wait
    FROM silver.fact_call_center_interactions
    GROUP BY ALL
""")

## 9. `sentiment_score` -- range sanity (dictionary: -1 to 1) + cross-check against `detected_sentiment`

In [ ]:
q("""
    SELECT
        typeof(sentiment_score) AS silver_type,
        min(sentiment_score) AS min_score, max(sentiment_score) AS max_score,
        count(*) FILTER (WHERE sentiment_score < -1 OR sentiment_score > 1) AS out_of_range,
        count(*) FILTER (WHERE sentiment_score IS NOT NULL) AS populated, count(*) AS total
    FROM silver.fact_call_center_interactions
    GROUP BY ALL
""")

In [ ]:
# Does a higher sentiment_score line up with a more positive detected_sentiment?
q("""
    SELECT detected_sentiment, count(*) AS n,
           round(avg(sentiment_score), 3) AS avg_score,
           round(min(sentiment_score), 3) AS min_score, round(max(sentiment_score), 3) AS max_score
    FROM silver.fact_call_center_interactions
    WHERE sentiment_score IS NOT NULL
    GROUP BY 1
    ORDER BY avg_score DESC
""")

## 10. Boolean columns -- cast correctness + cross-field logic

`was_resolved`, `requires_followup`, `was_escalated`, `has_transcript`, `has_recording`.

In [ ]:
q("""
    SELECT
        typeof(was_resolved) AS t1, typeof(requires_followup) AS t2, typeof(was_escalated) AS t3,
        typeof(has_transcript) AS t4, typeof(has_recording) AS t5,
        count(*) FILTER (WHERE was_resolved) AS resolved_true,
        count(*) FILTER (WHERE was_resolved IS NULL) AS resolved_null,
        count(*) FILTER (WHERE requires_followup) AS followup_true,
        count(*) FILTER (WHERE was_escalated) AS escalated_true,
        count(*) FILTER (WHERE has_transcript) AS transcript_true,
        count(*) FILTER (WHERE has_recording) AS recording_true,
        count(*) AS total
    FROM silver.fact_call_center_interactions
    GROUP BY ALL
""")

In [ ]:
# Does was_resolved = False correlate with requires_followup = True, as you'd expect operationally?
q("""
    SELECT was_resolved, requires_followup, count(*) AS n
    FROM silver.fact_call_center_interactions
    GROUP BY 1, 2
    ORDER BY 1, 2
""")

## 11. `customer_detected_accent`/`agent_used_accent` -- domain spot check

In [ ]:
print("customer_detected_accent:")
display(q("SELECT customer_detected_accent, count(*) AS n FROM bronze.call_center_interactions GROUP BY 1 ORDER BY n DESC"))
print("agent_used_accent:")
display(q("SELECT agent_used_accent, count(*) AS n FROM bronze.call_center_interactions GROUP BY 1 ORDER BY n DESC"))

## 12. `mentioned_products` -- population rate + format spot check

In [ ]:
total = q("SELECT count(*) AS n FROM bronze.call_center_interactions")["n"][0]
populated = q("SELECT count(*) AS n FROM bronze.call_center_interactions WHERE mentioned_products IS NOT NULL")["n"][0]
print(f"mentioned_products populated: {populated:,} / {total:,} = {100*populated/total:.2f}%")
print()
print("Sample values:")
display(q_full("SELECT mentioned_products FROM bronze.call_center_interactions WHERE mentioned_products IS NOT NULL LIMIT 10"))

## 13. `interaction_date`/`process_date` -- consistency + window check

Same checks run on `transactions`' `transaction_date`/`process_date` pair, since that table showed a real ~25% one-day rollback pattern worth re-checking here.

In [ ]:
q("""
    SELECT
        typeof(interaction_date) AS interaction_date_type, typeof(process_date) AS process_date_type,
        min(interaction_date) AS earliest_interaction, max(interaction_date) AS latest_interaction,
        min(process_date) AS earliest_process, max(process_date) AS latest_process,
        count(*) FILTER (WHERE process_date <> CAST(interaction_date AS DATE)) AS process_date_differs,
        count(*) FILTER (WHERE process_date < CAST(interaction_date AS DATE)) AS processed_before,
        count(*) FILTER (WHERE process_date > CAST(interaction_date AS DATE)) AS processed_after
    FROM silver.fact_call_center_interactions
    GROUP BY ALL
""")

In [ ]:
# Same window check as every other table.
q("""
    SELECT
        count(*) FILTER (WHERE interaction_date < TIMESTAMP '2023-06-17') AS before_window,
        count(*) FILTER (WHERE interaction_date > TIMESTAMP '2026-06-17 23:59:59') AS after_window
    FROM silver.fact_call_center_interactions
""")

## Checklist

- [ ] Column inventory (1): all 21 Silver columns present, types correct
- [ ] Row reconciliation (2): 800,000 rows as the dictionary declares, dedup clean
- [ ] NOT NULL audit (3): all 12 declared columns populated -- note whether `has_recording` is clean (fix candidate)
- [ ] Nullable columns (4): population rates make sense
- [ ] Domain checks (5): `interaction_type`/`reason_category`/`detected_sentiment` real values, `channel` gate re-verified
- [ ] FK integrity (6): `customer_id`/`agent_id` orphan rates reproduced directly
- [ ] `contact_reason` vs. `reason_category` (7): resolves the open question from `table_specs.py`'s comment -- are they really identical row-for-row?
- [ ] `duration_seconds`/`wait_time_seconds` (8): no negatives
- [ ] `sentiment_score` (9): within -1 to 1, lines up sensibly with `detected_sentiment`
- [ ] Boolean columns (10): cast correctness, `was_resolved`/`requires_followup` shape makes sense
- [ ] Accent columns (11): domain sanity
- [ ] `mentioned_products` (12): population/format sanity
- [ ] Dates (13): `process_date` vs. `interaction_date` consistency, window check

In [ ]:
con.close()
print("closed")